In [12]:
import numpy as np
import cv2

CLASS_NAMES = ["katun", "rayon", "linen", "mori"]

def postprocess_yolov8(output, input_shape, orig_shape, conf_threshold=0.5):
    """
    output      : raw tensor from interpreter, shape (1, 8, 8400)
    input_shape : (height, width) used during inference, e.g. (224, 224)
    orig_shape  : (height, width) of the original image
    """
    predictions = np.squeeze(output[0]).T   # (8400, 8)

    boxes_raw    = predictions[:, :4]       # cx, cy, w, h (normalized)
    class_scores = predictions[:, 4:]       # (8400, 4)

    class_ids    = np.argmax(class_scores, axis=1)
    confidences  = np.max(class_scores, axis=1)

    # Filter low-confidence detections
    mask = confidences > conf_threshold
    boxes_raw    = boxes_raw[mask]
    class_ids    = class_ids[mask]
    confidences  = confidences[mask]

    if len(boxes_raw) == 0:
        return [], [], []

    # Convert cx,cy,w,h → x1,y1,x2,y2 (still in input_shape scale)
    ih, iw = input_shape
    oh, ow = orig_shape

    cx, cy, w, h = boxes_raw[:, 0], boxes_raw[:, 1], boxes_raw[:, 2], boxes_raw[:, 3]
    x1 = (cx - w / 2) * ow
    y1 = (cy - h / 2) * oh
    x2 = (cx + w / 2) * ow
    y2 = (cy + h / 2) * oh

    boxes_xyxy = np.stack([x1, y1, x2, y2], axis=1).astype(int)

    # Non-Maximum Suppression to remove duplicate boxes
    indices = cv2.dnn.NMSBoxes(
        boxes_xyxy.tolist(), confidences.tolist(),
        score_threshold=conf_threshold, nms_threshold=0.45
    )
    indices = indices.flatten() if len(indices) > 0 else []

    return boxes_xyxy[indices], class_ids[indices], confidences[indices]


def draw_detections(image_path, boxes, class_ids, confidences):
    img = cv2.imread(image_path)
    for box, cls_id, conf in zip(boxes, class_ids, confidences):
        x1, y1, x2, y2 = box
        label = f"{CLASS_NAMES[cls_id]}: {conf:.2f}"
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(img, label, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    return img


# ── Main inference loop ──────────────────────────────────────────────
for file_name in image_files:
    image_path = os.path.join(test_image_dir, file_name)

    orig_img   = cv2.imread(image_path)
    orig_shape = orig_img.shape[:2]          # (H, W)

    input_data = preprocess_image(image_path, input_size=input_shape)
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()
    output_data = interpreter.get_tensor(output_details[0]['index'])

    boxes, class_ids, confidences = postprocess_yolov8(
        output_data,
        input_shape=input_shape,
        orig_shape=orig_shape,
        conf_threshold=0.5
    )

    print(f"\n{file_name}")
    if len(boxes) == 0:
        print("  → No detections above threshold")
    else:
        for box, cls_id, conf in zip(boxes, class_ids, confidences):
            print(f"  → {CLASS_NAMES[cls_id]:8s}  conf={conf:.2f}  box={box.tolist()}")

    # Optional: save annotated image
    result_img = draw_detections(image_path, boxes, class_ids, confidences)
    cv2.imwrite(f"./results/{file_name}", result_img)


IMG20251027122331_jpg.rf.c56b66cdd3158cac51c8256e6e9c6980.jpg
  → katun     conf=0.89  box=[-2, -1, 523, 524]

IMG20251027122613_jpg.rf.3d8d123501916daac95d22412ea811c4.jpg
  → linen     conf=0.89  box=[-1, 0, 524, 523]

IMG20251027122837_jpg.rf.7200be7575ec449bae145f3bc8fd4c66.jpg
  → linen     conf=0.68  box=[-3, 0, 525, 521]

IMG20251027122847_jpg.rf.cc6a75dcfd655504ec4bf102d0779f69.jpg
  → linen     conf=0.86  box=[-1, 0, 524, 524]

IMG20251027122855_jpg.rf.9dac916ec0deeae9a2af5d04209b5a87.jpg
  → linen     conf=0.97  box=[0, 0, 484, 524]

IMG20251027131112_jpg.rf.d2871e0aabfc44d29d5292b061dd06ec.jpg
  → rayon     conf=0.97  box=[0, 0, 524, 521]

IMG20251027131130_jpg.rf.41e861545430e3d2d4235dbe694684e4.jpg
  → rayon     conf=0.90  box=[0, 1, 524, 523]

IMG20251027131250_jpg.rf.32459e7a7859b6147b69473273f544ad.jpg
  → linen     conf=0.87  box=[-1, -2, 524, 524]

IMG20251128130601_jpg.rf.cecf007fa3b89eb4f62c9b9aecfccfb9.jpg
  → linen     conf=0.94  box=[-3, 0, 524, 522]

IMG2025112